In [ ]:
import numpy as np
import random

# 第15章 PPO算法

## 目录
1. PPO原理
2. 裁剪目标函数
3. 编程题

---
## 1. PPO原理

In [ ]:
"""PPO (Proximal Policy Optimization)"""
def ppo_loss(old_log_pi, new_log_pi, advantage, clip_eps=0.2):
    ratio = np.exp(new_log_pi - old_log_pi)
    clipped = np.clip(ratio, 1-clip_eps, 1+clip_eps)
    return -min(ratio * advantage, clipped * advantage)

class PPO:
    """PPO算法"""
    def __init__(self, n_states, n_actions, gamma=0.99, clip_eps=0.2):
        self.actor = np.zeros((n_states, n_actions))
        self.critic = np.zeros(n_states)
        self.gamma = gamma
        self.clip_eps = clip_eps
    
    def get_policy(self, s):
        exp_a = np.exp(self.actor[s] - np.max(self.actor[s]))
        return exp_a / np.sum(exp_a)
    
    def get_log_prob(self, s, a):
        pi = self.get_policy(s)
        return np.log(pi[a] + 1e-8)
    
    def update(self, trajectory, advantages):
        for (s, a), adv in zip(trajectory, advantages):
            old_log = self.get_log_prob(s, a)
            new_log = old_log
            ratio = np.exp(new_log - old_log)
            clipped = np.clip(ratio, 1-self.clip_eps, 1+self.clip_eps)
            actor_loss = -min(ratio * adv, clipped * adv)
            self.actor[s, a] -= 0.001 * actor_loss
            self.critic[s] += 0.01 * adv

print("PPO: 裁剪策略更新范围保证稳定性")

---
## 2. 编程题

### 编程题1：实现PPO算法在CartPole中的训练

In [ ]:
class SimpleCartPole:
    """简化CartPole"""
    def __init__(self):
        self.state = np.zeros(4)
    def reset(self): self.state = np.random.randn(4) * 0.1; return self.state.copy()
    def step(self, action):
        x, x_dot, theta, theta_dot = self.state
        force = 10.0 if action == 1 else -10.0
        costheta, sintheta = np.cos(theta), np.sin(theta)
        temp = force + 0.1 * 0.5 * theta_dot ** 2 * sintheta
        theta_acc = (9.8 * sintheta - costheta * temp) / (0.5 * (4.0/3.0 - 0.1 * costheta ** 2))
        x_acc = temp - 0.1 * theta_acc * costheta
        x += 0.02 * x_dot + 0.02 * x_acc
        x_dot += 0.02 * x_acc
        theta += 0.02 * theta_dot + 0.02 * theta_acc
        theta_dot += 0.02 * theta_acc
        self.state = np.array([x, x_dot, theta, theta_dot])
        done = x < -2.4 or x > 2.4 or theta < -0.2095 or theta > 0.2095
        return self.state.copy(), (0 if done else 1.0), done

class PPOAgent:
    """PPO智能体"""
    def __init__(self, n_states=8, gamma=0.99, clip_eps=0.2):
        self.n_states = n_states
        self.actor = np.zeros((n_states, 2))
        self.critic = np.zeros(n_states)
        self.gamma = gamma
        self.clip_eps = clip_eps
    
    def get_state(self, s):
        return int(np.clip(s[0] * 2 + 4, 0, self.n_states - 1))
    
    def get_policy(self, state):
        s = self.get_state(state)
        exp_a = np.exp(self.actor[s] - np.max(self.actor[s]))
        return exp_a / np.sum(exp_a)
    
    def select_action(self, state):
        return np.random.choice(2, p=self.get_policy(state))
    
    def update(self, trajectory, rewards):
        advantages = []
        G = 0
        for r in reversed(rewards):
            G = self.gamma * G + r
            advantages.insert(0, G)
        
        for (s, a), adv in zip(trajectory, advantages):
            s_idx = self.get_state(s)
            target = adv
            td = target - self.critic[s_idx]
            self.critic[s_idx] += 0.01 * td
            
            pi = self.get_policy(s)
            ratio = pi[a]
            clipped = np.clip(ratio, 1-self.clip_eps, 1+self.clip_eps)
            loss = -min(ratio * adv, clipped * adv)
            self.actor[s_idx, a] -= 0.001 * loss * (1 - pi[a])

def train_ppo():
    """训练PPO"""
    agent = PPOAgent()
    env = SimpleCartPole()
    rewards_history = []
    
    for ep in range(200):
        s = env.reset()
        trajectory, rewards = [], []
        
        for step in range(500):
            a = agent.select_action(s)
            ns, r, done = env.step(a)
            trajectory.append((s.copy(), a))
            rewards.append(r)
            s = ns
            if done: break
        
        rewards_history.append(sum(rewards))
        agent.update(trajectory, rewards)
    
    print("PPO训练结果:")
    print(f"  最终回报: {rewards_history[-1]}")
    print(f"  平均回报(最后10局): {np.mean(rewards_history[-10:]):.1f}")

train_ppo()

---

### 编程题2：实现PPO与Actor-Critic的性能对比

In [ ]:
class SimpleAC:
    """简化Actor-Critic"""
    def __init__(self, n_states=8):
        self.actor = np.zeros((n_states, 2))
        self.critic = np.zeros(n_states)
        self.gamma = 0.99
    
    def get_state(self, s):
        return int(np.clip(s[0] * 2 + 4, 0, 7))
    
    def select_action(self, state):
        s = self.get_state(state)
        exp_a = np.exp(self.actor[s] - np.max(self.actor[s]))
        return np.random.choice(2, p=exp_a / np.sum(exp_a))
    
    def update(self, s, a, r, ns, done):
        s_idx = self.get_state(s)
        ns_idx = self.get_state(ns)
        target = r + self.gamma * (0 if done else self.critic[ns_idx])
        advantage = target - self.critic[s_idx]
        self.critic[s_idx] += 0.01 * advantage
        pi = np.exp(self.actor[s_idx]) / np.sum(np.exp(self.actor[s_idx]))
        self.actor[s_idx, a] += 0.001 * advantage * (1 - pi[a])

def compare_ppo_ac():
    """对比PPO与AC"""
    ppo_agent = PPOAgent()
    ac_agent = SimpleAC()
    env = SimpleCartPole()
    
    ppo_rewards, ac_rewards = [], []
    
    for ep in range(200):
        for agent in [ppo_agent, ac_agent]:
            s = env.reset()
            traj, rews = [], []
            for _ in range(500):
                a = agent.select_action(s) if hasattr(agent, 'select_action') else agent.select_action(s)
                ns, r, done = env.step(a)
                if hasattr(agent, 'update'):
                    traj.append((s.copy(), a))
                    rews.append(r)
                else: agent.update(s, a, r, ns, done)
                s = ns
                if done: break
            if hasattr(agent, 'update'): agent.update(traj, rews)
            if agent == ppo_agent: ppo_rewards.append(sum(rews))
            else: ac_rewards.append(sum(rews))
    
    print("PPO vs Actor-Critic对比:")
    print(f"  PPO平均回报: {np.mean(ppo_rewards[-20:]):.1f}")
    print(f"  AC平均回报: {np.mean(ac_rewards[-20:]):.1f}")

compare_ppo_ac()

---

### 编程题3：实现自适应裁剪范围的PPO

In [ ]:
class AdaptivePPO:
    """自适应裁剪PPO"""
    def __init__(self, n_states, initial_eps=0.3, target_kl=0.01):
        self.actor = np.zeros((n_states, 2))
        self.critic = np.zeros(n_states)
        self.clip_eps = initial_eps
        self.target_kl = target_kl
        self.gamma = 0.99
    
    def get_kl(self, s, old_pi, new_pi):
        return np.sum(old_pi * np.log(old_pi / (new_pi + 1e-8) + 1e-8))
    
    def update(self, s, a, old_pi, advantage):
        s_idx = s % len(self.actor)
        exp_a = np.exp(self.actor[s_idx] - np.max(self.actor[s_idx]))
        pi = exp_a / np.sum(exp_a)
        
        ratio = pi[a] / (old_pi[a] + 1e-8)
        clipped = np.clip(ratio, 1-self.clip_eps, 1+self.clip_eps)
        loss = -min(ratio * advantage, clipped * advantage)
        
        kl = self.get_kl(s_idx, old_pi, pi)
        if kl > self.target_kl * 1.5: self.clip_eps *= 0.9
        elif kl < self.target_kl * 0.5: self.clip_eps *= 1.1
        self.clip_eps = np.clip(self.clip_eps, 0.05, 0.5)
        
        self.actor[s_idx, a] -= 0.001 * loss
        self.critic[s_idx] += 0.01 * advantage

def test_adaptive_ppo():
    """测试自适应PPO"""
    agent = AdaptivePPO(n_states=8)
    env = SimpleCartPole()
    clip_history = []
    
    for ep in range(100):
        s = env.reset()
        old_pi = np.array([0.5, 0.5])
        total = 0
        for _ in range(500):
            a = np.random.choice(2, p=old_pi)
            ns, r, done = env.step(a)
            advantage = r - 0.5
            agent.update(s, a, old_pi, advantage)
            s = ns
            total += r
            old_pi = agent.actor[0] - np.max(agent.actor[0])
            old_pi = old_pi / np.sum(old_pi)
            if done: break
        clip_history.append(agent.clip_eps)
    
    print("自适应PPO:")
    print(f"  初始裁剪范围: {clip_history[0]:.3f}")
    print(f"  最终裁剪范围: {clip_history[-1]:.3f}")

test_adaptive_ppo()

print("\n第15章 PPO算法 - 编程题完成!")